## E2 — Attribute controllability (Kaggle)\n
\n
Quantitative controllability test for all models:\n
- Plain Transformer\n
- Simple VAE\n
- Hierarchical VAE\n
\n
For each attribute (polyphony, intensity, velocity, density) and each requested bin 0–7:\n
- generate `N` samples\n
- recompute the raw attribute statistics from the generated output\n
- re-bin using the saved global quantile edges (`artifacts/attributes/quantiles.json`)\n
\n
Reports:\n
- Pearson + Spearman correlation (requested bin vs measured raw stat)\n
- exact-bin and ±1-bin accuracy\n
- confusion matrices\n
- disentanglement check (non-target attributes vs target bin)\n
\n
Outputs: `artifacts/eval_controllability/`\n

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

IS_KAGGLE = Path('/kaggle').exists() and Path('/kaggle/input').exists()
INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else Path.cwd().parents[0]
WORKING_ROOT = Path('/kaggle/working') if IS_KAGGLE else (Path.cwd() / 'artifacts' / 'local_eval')
OUT_DIR = (WORKING_ROOT / 'artifacts' / 'eval_controllability').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR:', OUT_DIR)

if IS_KAGGLE:
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / 'ml' / 'src' / 'musicgen').is_dir():
            return p
    if IS_KAGGLE:
        for d in sorted(INPUT_ROOT.iterdir()):
            if not d.is_dir():
                continue
            if (d / 'ml' / 'src' / 'musicgen').is_dir():
                return d
            for sub in d.iterdir():
                if sub.is_dir() and (sub / 'ml' / 'src' / 'musicgen').is_dir():
                    return sub
    raise RuntimeError('Could not locate repo root containing ml/src/musicgen. Add the repo as a Kaggle dataset.')


def _find_path(rel: str) -> Path:
    cand = (REPO_ROOT / rel).resolve()
    if cand.exists():
        return cand
    if IS_KAGGLE:
        for d in sorted(INPUT_ROOT.iterdir()):
            if not d.is_dir():
                continue
            cand2 = (d / rel).resolve()
            if cand2.exists():
                return cand2
            for sub in d.iterdir():
                if sub.is_dir():
                    cand3 = (sub / rel).resolve()
                    if cand3.exists():
                        return cand3
    raise FileNotFoundError(f'Missing required path: {rel} (checked repo + /kaggle/input/*)')


REPO_ROOT = _find_repo_root()
print('REPO_ROOT:', REPO_ROOT)

# Repo imports
import sys
ml_src = (REPO_ROOT / 'ml' / 'src').resolve()
if str(ml_src) not in sys.path:
    sys.path.insert(0, str(ml_src))

from musicgen.inference.service import GenerateRequest
from musicgen.inference.tokenizer import load_remi_tokenizer
from musicgen.inference.generate import generate_remi_tokens, GenerationConfig
from musicgen.eval.attributes import (
    ATTRIBUTE_NAMES,
    bin_attributes,
    compute_raw_attributes_per_bar_from_tokens,
    load_quantile_thresholds,
)

TOK = load_remi_tokenizer()
QUANTILES_PATH = _find_path('artifacts/attributes/quantiles.json')
print('QUANTILES_PATH:', QUANTILES_PATH)
THRESHOLDS = load_quantile_thresholds(QUANTILES_PATH)
BAR_ID = int(TOK.bar_id)
ID_TO_TOKEN = TOK.id_to_token


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    model_type: str
    ckpt_path: str  # absolute path on Kaggle
    block_size: int = 1024
    bars_per_sample: int = 8


def _ckpt(rel: str) -> str:
    return str(_find_path(rel))


SPECS = [
    ModelSpec('plain_v3', 'plain', _ckpt('ml/src/musicgen/runs/plain_v3/ckpt.pt'), 1024),
    ModelSpec('simple_v2', 'simple_vae', _ckpt('ml/src/musicgen/runs/simple_v2/ckpt.pt'), 1024),
    ModelSpec('vae_v3', 'vae', _ckpt('ml/src/musicgen/runs/vae_v3/ckpt.pt'), 1024),
]

# Controllability run config
N_PER_BIN = 50
BASELINE_BIN = 3
TEMP = 1.0
TOP_P = 0.9

print('N_PER_BIN:', N_PER_BIN)
print('baseline bin:', BASELINE_BIN)
for s in SPECS:
    print('ckpt:', s.name, s.ckpt_path)


In [ ]:
def rank_correlation(x: list[float], y: list[float]) -> float:
    # Spearman via rank transform (no scipy dependency)
    rx = np.argsort(np.argsort(np.asarray(x)))
    ry = np.argsort(np.argsort(np.asarray(y)))
    if rx.std() == 0 or ry.std() == 0:
        return float('nan')
    return float(np.corrcoef(rx, ry)[0, 1])


def pearson(x: list[float], y: list[float]) -> float:
    xx = np.asarray(x, dtype=float)
    yy = np.asarray(y, dtype=float)
    if xx.std() == 0 or yy.std() == 0:
        return float('nan')
    return float(np.corrcoef(xx, yy)[0, 1])


def accuracy(y_true: list[int], y_pred: list[int], tol: int = 0) -> float:
    a = np.asarray(y_true, dtype=int)
    b = np.asarray(y_pred, dtype=int)
    return float(np.mean(np.abs(a - b) <= int(tol)))


def confusion(y_true: list[int], y_pred: list[int], n: int = 8) -> list[list[int]]:
    m = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < n and 0 <= p < n:
            m[int(t), int(p)] += 1
    return m.tolist()


In [ ]:
# Core loop: generate tokens (not just MIDI) so we can recompute attributes from tokens.

def generate_tokens_for_request(*, req: GenerateRequest) -> list[int]:
    # Use the same service model loader/caching by calling generate_midi_bytes is opaque.
    # For controllability we need tokens, so we re-run the inference loop here.
    from musicgen.inference.service import _load_checkpoint as _load_ckpt  # type: ignore
    from musicgen.inference.service import _build_model as _build_model  # type: ignore
    from musicgen.inference.service import _resolve_ckpt_path as _resolve_ckpt_path  # type: ignore

    ckpt_path = _resolve_ckpt_path(req.ckpt_path or '')
    ckpt = _load_ckpt(ckpt_path)
    model = _build_model(req.model_type, ckpt, block_size=req.block_size, bars=req.bars_per_sample)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    attrs = torch.tensor(req.attributes, dtype=torch.long, device=device).unsqueeze(0)

    gen_cfg = GenerationConfig(
        block_size=int(req.block_size),
        bars_per_sample=int(req.bars_per_sample),
        temperature=float(req.temperature),
        top_p=float(req.top_p),
    )

    tokens = generate_remi_tokens(
        model_type=req.model_type,
        model=model,
        attributes=attrs,
        pad_id=TOK.pad_id,
        bos_id=TOK.bos_id,
        bar_id=TOK.bar_id,
        id_to_token=TOK.id_to_token,
        cfg=gen_cfg,
        device=device,
        z_p=None,
        seed=int(req.seed),
    )
    return tokens


def uniform_attrs(bin4: list[int], bars: int = 8) -> list[list[int]]:
    return [list(map(int, bin4)) for _ in range(bars)]


In [ ]:
report: dict[str, Any] = {'config': {'N_PER_BIN': N_PER_BIN, 'BASELINE_BIN': BASELINE_BIN, 'TEMP': TEMP, 'TOP_P': TOP_P}, 'models': {}}

for spec in SPECS:
    model_res: dict[str, Any] = {}
    print('===', spec.name, '===')
    for a_idx, a_name in enumerate(ATTRIBUTE_NAMES):
        # Collect measurements
        req_bins: list[int] = []
        meas_bins: list[int] = []
        meas_raw: list[float] = []
        other_raw: list[list[float]] = []

        for b in range(8):
            # baseline 4-vector
            v = [BASELINE_BIN, BASELINE_BIN, BASELINE_BIN, BASELINE_BIN]
            v[a_idx] = int(b)
            attrs = uniform_attrs(v, bars=spec.bars_per_sample)

            for _ in range(N_PER_BIN):
                seed = random.randint(0, 2**31 - 1)
                req = GenerateRequest(
                    model_type=spec.model_type,  # type: ignore[arg-type]
                    attributes=attrs,
                    seed=int(seed),
                    temperature=float(TEMP),
                    top_p=float(TOP_P),
                    block_size=int(spec.block_size),
                    bars_per_sample=int(spec.bars_per_sample),
                    ckpt_path=spec.ckpt_path,
                )

                tokens = generate_tokens_for_request(req=req)
                raw_bars = compute_raw_attributes_per_bar_from_tokens(
                    tokens,
                    id_to_token=ID_TO_TOKEN,
                    bar_token_id=BAR_ID,
                    bars_per_sample=spec.bars_per_sample,
                )
                raw_arr = np.asarray(raw_bars, dtype=float)
                # Reduce to a single scalar per attribute by mean over bars
                raw_mean4 = raw_arr.mean(axis=0)

                bins_bars = bin_attributes(raw_bars, thresholds=THRESHOLDS, bars_per_sample=spec.bars_per_sample)
                bins_arr = np.asarray(bins_bars, dtype=int)
                bins_mean4 = np.rint(bins_arr.mean(axis=0)).astype(int)

                req_bins.append(int(b))
                meas_bins.append(int(bins_mean4[a_idx]))
                meas_raw.append(float(raw_mean4[a_idx]))
                other_raw.append([float(raw_mean4[j]) for j in range(4)])

        pear = pearson([float(x) for x in req_bins], [float(x) for x in meas_raw])
        spear = rank_correlation([float(x) for x in req_bins], [float(x) for x in meas_raw])
        acc0 = accuracy(req_bins, meas_bins, tol=0)
        acc1 = accuracy(req_bins, meas_bins, tol=1)

        # Disentanglement: correlations of non-target attributes vs requested bin
        other = np.asarray(other_raw, dtype=float)
        disent: dict[str, float] = {}
        for j, nm in enumerate(ATTRIBUTE_NAMES):
            if j == a_idx:
                continue
            disent[nm] = pearson([float(x) for x in req_bins], other[:, j].tolist())

        model_res[a_name] = {
            'pearson': pear,
            'spearman': spear,
            'acc_exact': acc0,
            'acc_pm1': acc1,
            'confusion': confusion(req_bins, meas_bins, n=8),
            'disentanglement_pearson': disent,
        }

    report['models'][spec.name] = model_res

out_path = OUT_DIR / 'controllability_report.json'
out_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
out_path
